In [1]:
println("Loading imports")
include("../src/optim/gradient_descent.jl")
include("../src/optim/greedy_descent.jl")
include("../src/optim/loss.jl")

Loading imports


MPE_loss_init (generic function with 1 method)

Consider the following real-valued, parametrised function
\begin{align*}
f_a: \mathbb{Q}_2 & \rightarrow \mathbb{R} \cup \{\infty\}\\
x & \mapsto |(x-a)(x-2a)(x-4a)|_2
\end{align*}
where $\mathbb{Q}_2$ are the 2-adic numbers, $|\cdot|_2$ is the 2-adic norm, $a$ is the parameter, and $x$ is the variable.


Fix data points $X = \{1,2,4\} \subset \mathbb{Q}_2$ and consider the following loss function:
\begin{align*}
\ell: \mathbb{Q}_2 & \rightarrow \mathbb{R} \cup \{\infty\}\\
a & \mapsto \sum_{x \in X} f_a(x)
\end{align*}

Clearly, $a=1$ minimizes $\ell$ with $\ell(1)=0$.  In the Berkovich
analytification of $\mathbb{Q}_2$, can you trace a path from $a=8$ to $a=1$ whilst
only going in a direction with negative directional derivative of $\ell$?

In [2]:
println("Initialising data")
prec = 20
K = PadicField(2, prec)

# Set up the 3 data points in the 2-adic numbers
c1 = [K(1)]
r1 = Vector{Int}([prec])
c2 = [K(2)]
r2 = Vector{Int}([prec])
c3 = [K(4)]
r3 = Vector{Int}([prec])
p1 = ValuationPolydisc(c1, r1) 
p2 = ValuationPolydisc(c2, r2) 
p3 = ValuationPolydisc(c2, r2)

# We're trying to fit the data points (x, y) = {(1, 0), (2, 0), (4, 0)}
# using the function g_a(x) = (x-a)(x-2a)(x-4a)
data = [(p1, 0), (p2, 0), (p3, 0)]
R, (x, a) =  polynomial_ring(K, ["x", "a"])

# Initialise g_a(x) using a wrapper for absolute polynomials and their sums
g = PolydiscFunction([(x-a)*(x-2*a)*(x-4*a)])
# Specify that x is the data variable and a is the parameter variable
f = AbstractModel(g, [true, false])
# ℓ is the loss function of this model wrt the L¹ norm 
ℓ = MPE_loss_init(1)

Initialising data


Loss(var"#MPE_compute#125"{Int64}(1), var"#MPE_grad#127"{Int64}(1))

In [3]:
println("Initialising optimisation tools")
# Let's pick the Gauss point as our initial parameter
model = Model(f, ValuationPolydisc([K(1)], [0]))
# We optimise using Greedy descent
greedy_optim = greedy_descent_init(data, model, ℓ)

Initialising optimisation tools


OptimSetup{PadicFieldElem, Int64, Int64}(Tuple{ValuationPolydisc{PadicFieldElem, Int64}, Int64}[(ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^0 + O(2^20)], [20]), 0), (ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^1 + O(2^21)], [20]), 0), (ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^1 + O(2^21)], [20]), 0)], Model{PadicFieldElem, Int64}(AbstractModel{PadicFieldElem}(PolydiscFunction{PadicFieldElem}(AbstractAlgebra.Generic.MPoly{PadicFieldElem}[x^3 + (2^0 + 2^3 + 2^4 + 2^5 + 2^6 + 2^7 + 2^8 + 2^9 + 2^10 + 2^11 + 2^12 + 2^13 + 2^14 + 2^15 + 2^16 + 2^17 + 2^18 + 2^19 + O(2^20))*x^2*a + (2^1 + 2^2 + 2^3 + O(2^21))*x*a^2 + (2^3 + 2^4 + 2^5 + 2^6 + 2^7 + 2^8 + 2^9 + 2^10 + 2^11 + 2^12 + 2^13 + 2^14 + 2^15 + 2^16 + 2^17 + 2^18 + 2^19 + 2^20 + 2^21 + 2^22 + O(2^23))*a^3]), Bool[1, 0]), ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^0 + O(2^20)], [0])), Loss(var"#MPE_compute#125"{Int64}(1), var"#MPE_grad#127"{Int64}(1)), var"#115#116"{Loss, Int6

In [4]:
println("Optimising parameter")
# Compute the loss 

println("Loss before starting greedy descent is ", eval_loss(greedy_optim))

# Make 20 steps using greedy descent
N_epochs = 20
t1 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(greedy_optim))
    step!(greedy_optim)
end 
t2 = time()

# Compute the new value of the loss
println("Greedy descent finished in ", t2-t1, " seconds. \nThe Final parameter a is ", greedy_optim.model.param)

Optimising parameter
Loss before starting greedy descent is 0.5
Loss at epoch 1 is 0.5
Loss at epoch 2 is 0.25
Loss at epoch 3 is 0.125
Loss at epoch 4 is 0.0625
Loss at epoch 5 is 0.03125
Loss at epoch 6 is 0.015625
Loss at epoch 7 is 0.0078125
Loss at epoch 8 is 0.00390625
Loss at epoch 9 is 0.001953125
Loss at epoch 10 is 0.0009765625
Loss at epoch 11 is 0.00048828125
Loss at epoch 12 is 0.000244140625
Loss at epoch 13 is 0.0001220703125
Loss at epoch 14 is 6.103515625e-5
Loss at epoch 15 is 3.0517578125e-5
Loss at epoch 16 is 1.52587890625e-5
Loss at epoch 17 is 7.62939453125e-6
Loss at epoch 18 is 3.814697265625e-6
Loss at epoch 19 is 1.9073486328125e-6
Loss at epoch 20 is 9.5367431640625e-7
Greedy descent finished in 0.5865738391876221 seconds. 
The Final parameter a is ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^0 + O(2^20)], [20])


In [5]:
# Declare new model with same starting point as the first one
new_model = Model(f, ValuationPolydisc([K(1)], [0]))
# Now let's optimise using gradient descent 
gradient_optim = gradient_descent_init(data, new_model, ℓ)

println("Loss before starting gradient descent is ", eval_loss(gradient_optim))

t3 = time()
for i in 1:N_epochs
    println("Loss at epoch ", i, " is ", eval_loss(gradient_optim))
    step!(gradient_optim)
end 
t4 = time()

# Compute the new value of the loss
println("Gradient descent finished in ", t4-t3, "Seconds. \nThe Final parameter a is ", gradient_optim.model.param)


Loss before starting gradient descent is 0.5
Loss at epoch 1 is 0.5
Loss at epoch 2 is 0.25
Loss at epoch 3 is 0.125
Loss at epoch 4 is 0.0625
Loss at epoch 5 is 0.03125
Loss at epoch 6 is 0.015625
Loss at epoch 7 is 0.0078125
Loss at epoch 8 is 0.00390625
Loss at epoch 9 is 0.001953125
Loss at epoch 10 is 0.0009765625
Loss at epoch 11 is 0.00048828125
Loss at epoch 12 is 0.000244140625
Loss at epoch 13 is 0.0001220703125
Loss at epoch 14 is 6.103515625e-5
Loss at epoch 15 is 3.0517578125e-5
Loss at epoch 16 is 1.52587890625e-5
Loss at epoch 17 is 7.62939453125e-6
Loss at epoch 18 is 3.814697265625e-6
Loss at epoch 19 is 1.9073486328125e-6
Loss at epoch 20 is 9.5367431640625e-7
Gradient descent finished in 1.2538859844207764Seconds. 
The Final parameter a is ValuationPolydisc{PadicFieldElem, Int64}(PadicFieldElem[2^0 + O(2^20)], [20])
